# Mantenimiento y Gobierno de Datos en Delta Lake

Este notebook demuestra cómo Delta Lake permite optimizar, mantener y recuperar tablas en entornos de procesamiento distribuido utilizando Apache Spark y Databricks.

A lo largo del flujo se exploran problemas comunes de producción, como el fenómeno de *Small Files* generado por ingestas incrementales, y se aplican distintas estrategias de optimización y gobierno de datos.

Además, se muestran capacidades avanzadas de Delta Lake como:

- Compactación de archivos
- Data Skipping con ZORDER
- Limpieza de almacenamiento
- Auditoría de operaciones
- Time Travel
- Recuperación atómica de datos

## Capacidades principales de Delta Lake

| Categoría | Funcionalidad | Objetivo |
|---|---|---|
| Diagnóstico | DESCRIBE DETAIL | Analizar estructura y fragmentación |
| Auditoría | DESCRIBE HISTORY | Consultar historial y versiones |
| Compactación | OPTIMIZE | Reducir Small Files |
| Optimización analítica | ZORDER | Mejorar Data Skipping |
| Higiene de almacenamiento | VACUUM | Eliminar archivos obsoletos |
| Time Travel | VERSION / TIMESTAMP AS OF | Consultar snapshots históricos |
| Recuperación | RESTORE TABLE | Restaurar versiones anteriores |

## El problema de los Small Files

En arquitecturas modernas de datos es común trabajar con:

- cargas incrementales
- micro-batches
- pipelines streaming
- escrituras frecuentes tipo append

Cada operación genera nuevos archivos Parquet.

Con el tiempo esto produce fragmentación y el fenómeno conocido como *Small Files*, donde una tabla termina compuesta por cientos o miles de archivos pequeños.

Esto impacta negativamente en Spark debido a:

- mayor metadata overhead
- incremento de operaciones I/O
- mayor carga sobre el driver
- degradación del rendimiento de lectura

# Creación de la tabla Delta

Para la demostración utilizaremos el dataset público NYC Taxi disponible en Databricks.

El objetivo es simular un entorno realista donde posteriormente aplicaremos:

- ingestas incrementales
- compactación
- optimización
- recuperación histórica

In [0]:
%sql
CREATE TABLE workspace.prueba.nyctaxi_prueba
USING DELTA
AS
SELECT *
FROM csv.`/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz`

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

## Estado inicial

| Métrica | Valor |
|---|---|
| Número de archivos | 1 |
| Tamaño aproximado | ~100 MB |
| Particiones | No |
| Clustering | No |
| Formato | Delta |
| Compresión | ZSTD |

## Observaciones

La tabla se encuentra inicialmente en un estado óptimo:

- existe un único archivo Parquet
- no hay fragmentación
- no existen problemas de Small Files
- las lecturas son eficientes

Además, Delta Lake incorpora funcionalidades modernas como:

- `deletionVectors`
- Time Travel
- transaction log
- versionado automático

## Conclusión

Dado que la tabla ya se encuentra optimizada, será necesario simular múltiples escrituras incrementales para recrear un escenario real de fragmentación.
 

# Simulación de Ingesta Incremental

En entornos productivos es común trabajar con:

- pipelines batch incrementales
- micro-batches
- cargas periódicas
- streaming

Cada operación `append` genera nuevos archivos Parquet dentro de la tabla Delta.

Con el tiempo, este patrón produce fragmentación y el fenómeno conocido como *Small Files*, donde la tabla queda compuesta por múltiples archivos pequeños.

El objetivo de esta sección es recrear artificialmente ese escenario para posteriormente aplicar estrategias de optimización.

In [0]:
df = spark.table("workspace.prueba.nyctaxi_prueba")

for i in range(20):
    df.sample(fraction=0.05).write.format("delta").mode("append").saveAsTable("workspace.prueba.nyctaxi_prueba")

#mode("append") no sobrescribe, agrega datos nuevos

## ¿Qué ocurre durante las escrituras append?

Cada ejecución:

- crea nuevos archivos Parquet
- actualiza el transaction log
- incrementa el número de archivos activos

Aunque este patrón es eficiente para ingestas incrementales, puede degradar el rendimiento analítico cuando la fragmentación crece excesivamente.

In [0]:
%sql
DESCRIBE DETAIL workspace.prueba.nyctaxi_prueba

# Estado de la tabla luego de la ingesta incremental

## Comparación del estado físico

| Métrica | Estado Inicial | Luego de append |
|---|---|---|
| Número de archivos | 1 | 46 |
| Tamaño aproximado | ~100 MB | ~290 MB |
| Fragmentación | Baja | Alta |

## Observaciones

La tabla pasó de un estado inicialmente optimizado a un escenario más realista de producción:

- múltiples archivos pequeños
- mayor fragmentación
- incremento del metadata overhead

Este fenómeno es conocido como *Small Files*.

## Impacto técnico

A medida que aumenta la cantidad de archivos:

- Spark debe abrir más archivos durante cada lectura
- crece el overhead sobre el driver
- aumentan las operaciones I/O
- el rendimiento de las consultas puede degradarse

## Configuración Delta

La tabla mantiene funcionalidades modernas de Delta Lake:

- compresión ZSTD
- deletionVectors habilitado
- transaction log
- versionado automático

## Próximo paso

Aplicaremos `OPTIMIZE` y `ZORDER` para:

- compactar archivos
- reducir fragmentación
- mejorar Data Skipping
- optimizar consultas analíticas

# Compactación y Optimización de Archivos

Luego de la ingesta incremental, la tabla presenta un escenario típico de fragmentación generado por múltiples escrituras append.

Antes de optimizar, inspeccionamos nuevamente el estado físico de la tabla.


In [0]:
%sql
optimize workspace.prueba.nyctaxi_prueba zorder by (_c1)

## ¿Qué realiza esta operación?

La instrucción ejecuta dos procesos complementarios:

### OPTIMIZE
Compacta múltiples archivos pequeños en un conjunto reducido de archivos de mayor tamaño.

### ZORDER
Reorganiza físicamente los datos utilizando la columna `_c1` para mejorar el Data Skipping en consultas con filtros.

In [0]:
%sql
describe detaIl workspace.prueba.nyctaxi_prueba

# Estado de la tabla luego de OPTIMIZE

## Comparación antes vs después

| Métrica | Antes | Después |
|---|---|---|
| Número de archivos | 46 | 4 |
| Tamaño aproximado | ~290 MB | ~294 MB |
| Fragmentación | Alta | Baja |

## Observaciones

La operación `OPTIMIZE` redujo drásticamente la cantidad de archivos activos:

:contentReference[oaicite:0]{index=0}

Esto elimina el problema de *Small Files* consolidando los datos en archivos más grandes y eficientes para lectura analítica.


## Impacto técnico de la compactación

La compactación mejora el rendimiento debido a que Spark necesita:

- abrir menos archivos
- gestionar menos metadata
- realizar menos operaciones I/O

Esto reduce el overhead sobre el driver y mejora la eficiencia de las consultas distribuidas.

## Importante

`OPTIMIZE` no modifica los datos lógicos de la tabla.

Únicamente reorganiza físicamente los archivos Parquet.

# ZORDER y Data Skipping

Delta Lake mantiene estadísticas por archivo sobre los valores almacenados en cada bloque de datos.

`ZORDER` reorganiza físicamente filas con valores similares para maximizar la eficiencia del *Data Skipping*.

Esto permite que Spark descarte archivos irrelevantes antes de leerlos.

## Casos ideales para ZORDER

Se recomienda utilizar columnas:

- de alta cardinalidad
- utilizadas frecuentemente en cláusulas `WHERE`
- presentes en consultas analíticas

## Ejemplos comunes

- user_id
- order_id
- timestamp
- fechas
- regiones

# Higiene de almacenamiento con VACUUM

Las operaciones `OPTIMIZE`, `DELETE` y `UPDATE` generan archivos obsoletos que ya no forman parte de la versión activa de la tabla.

Sin embargo, estos archivos continúan almacenados físicamente para permitir capacidades como:

- Time Travel
- auditoría histórica
- recuperación de datos

La operación `VACUUM` elimina definitivamente esos archivos antiguos para liberar espacio de almacenamiento.

In [0]:
%sql
vacuum workspace.prueba.nyctaxi_prueba

## Importante

`VACUUM` elimina archivos físicos que ya no son referenciados por el transaction log.

Una vez eliminados, las versiones antiguas dependientes de esos archivos dejan de estar disponibles para Time Travel o RESTORE.

# Historial de versiones y auditoría

Delta Lake registra todas las operaciones realizadas sobre la tabla en el transaction log.

Esto permite:
- auditoría
- trazabilidad
- recuperación
- versionado histórico

In [0]:
%sql
describe history workspace.prueba.nyctaxi_prueba

## Operaciones registradas

En el historial podemos observar operaciones como:

- WRITE
- OPTIMIZE
- DELETE
- VACUUM

Cada operación genera una nueva versión inmutable de la tabla. <br>


Delta Lake implementa un modelo basado en snapshots.

Cada transacción crea una nueva versión consistente de la tabla sin modificar las versiones anteriores.


# Time Travel y recuperación de datos

A continuación realizaremos un `DELETE` para demostrar cómo Delta Lake permite acceder a versiones históricas de la tabla.

In [0]:
df.count()

## Estado antes del DELETE

Cantidad de registros: **20345868**

In [0]:
%sql
delete from workspace.prueba.nyctaxi_prueba where _c1 between "2018-01-01" and "2019-01-01"

In [0]:
df.count()

## Estado después del DELETE

Cantidad de registros: **20344910**

La operación eliminó registros lógicamente, creando una nueva versión de la tabla.

## ¿Qué ocurre internamente?

Delta Lake no elimina inmediatamente los archivos antiguos.

En cambio:
- crea una nueva versión del transaction log
- marca los registros como eliminados
- mantiene versiones anteriores disponibles para Time Travel

In [0]:
%sql
describe history workspace.prueba.nyctaxi_prueba

La operación `DELETE` quedó registrada como una nueva versión de la tabla.

La versión anterior al DELETE corresponde a la versión 26.
# Consulta histórica con VERSION AS OF

Podemos consultar el estado exacto de la tabla antes del DELETE utilizando Time Travel.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.prueba.nyctaxi_prueba VERSION AS OF 26;

El resultado confirma que los registros eliminados siguen disponibles en la versión histórica de la tabla.

In [0]:
%sql
RESTORE TABLE workspace.prueba.nyctaxi_prueba TO VERSION AS OF 26

In [0]:
df = spark.table("workspace.prueba.nyctaxi_prueba")
df.count()

# Recuperación completada

La tabla fue restaurada exitosamente a una versión anterior utilizando `RESTORE TABLE`.

Esta operación es atómica y permite recuperar estados históricos completos de manera segura.

# Conclusiones

Delta Lake incorpora capacidades fundamentales para entornos modernos de Data Engineering:

- optimización automática
- compactación de archivos
- clustering inteligente
- versionado
- auditoría
- recuperación histórica
- gobierno de datos

Estas funcionalidades permiten construir pipelines más confiables, eficientes y resilientes sobre Apache Spark.